# Spanish Electricity Demand Forecasting with a Heterogeneous Ensemble

**UPF Machine Learning course project · 2026**

**Authors:** Didac Bustamante, Nil Castaño, Taiki Causton and Joan Garcia Almenara

This notebook investigates hour-ahead electricity-demand forecasting for Spain
using hourly energy-system and weather data from 2015 to 2018. It compares a
seasonal naive benchmark with seven statistical and machine-learning models and
then combines their forecasts using heterogeneous ensembles.

The target is `total load actual`, measured in MWh. Price variables are excluded
because this project forecasts physical electricity demand rather than market
prices.

## Data and attribution

The two CSV files come from the Kaggle dataset [Hourly energy demand, generation
and weather](https://www.kaggle.com/datasets/nicholasjhana/energy-consumption-generation-prices-and-weather).
Place them in `data/raw/` as described in the repository's data README.

The data-cleaning and exploratory structure was informed by Dimitrios Roussis's
[Electricity price forecasting with DNNs (+ EDA)](https://www.kaggle.com/code/dimitriosroussis/electricity-price-forecasting-with-dnns-eda),
released under the Apache 2.0 licence. This project changes the prediction target,
feature engineering, model comparison and ensemble methodology.


## 📦 Install & Import Libraries

## Environment

Install the dependencies from the repository root before running the notebook:

```bash
python -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
```


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from math import sqrt

import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

import holidays
from scipy.optimize import minimize

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import LinearSVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import xgboost as xgb


os.makedirs("../reports/figures", exist_ok=True)
SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 10,
})

print("✅ All libraries loaded")


For this project, we work with **two CSV files** that together provide a complete hourly snapshot of Spain's electricity system over a **four-year window (2015–2018)**:

`weather_features.csv`: Hourly meteorological measurements (temperature, humidity, wind speed, precipitation, qualitative weather descriptors, etc.) recorded for **five of Spain's largest cities**: Madrid, Barcelona, Valencia, Seville and Bilbao.

`energy_dataset.csv`: Hourly records of the Spanish electricity grid, including the **amount of electricity generated by each source** (with fossil gas, hard coal and wind being the dominant ones), the **total system load** (i.e. nationwide demand), and the **electricity price** in €/MWh. *Note: although every generation column is reported in MW and the data is hourly, each cell effectively represents **MWh** (Megawatt-hours).*

Restricting the weather information to these five cities is sufficient for our purposes: their geographic spread covers most of the Spanish mainland reasonably evenly, and together they account for roughly **one third of Spain's total population** — meaning their weather conditions are a strong proxy for the country-wide drivers of electricity demand.

## 1.1. Energy dataset

In [ ]:
df_energy = pd.read_csv("../data/raw/energy_dataset.csv", parse_dates=["time"])
print(f"Energy dataset shape: {df_energy.shape}")
df_energy.head()


In [ ]:
# Quick look at types and missing values
print("Missing values per column:")
print(df_energy.isnull().sum()[df_energy.isnull().sum() > 0].to_string())
print(f"\nDate range: {df_energy['time'].min()} → {df_energy['time'].max()}")


### Cleaning steps for the energy dataset

1. Drop columns that are 100% empty: `generation hydro pumped storage aggregated`, `forecast wind offshore eday ahead`.
2. Drop `price actual` and `price day ahead`: they were the target of the original Kaggle project, not ours. We exclude them to avoid conceptual leakage.
3. Convert `time` from UTC string to a datetime index.
4. **[Improvement S1]** Impute the target's missing values first with `shift(24)` (same hour previous day, preserves the daily pattern), and only then use linear interpolation as fallback.

In [ ]:
# Drop fully-null columns + price columns (we predict demand, not price)
cols_to_drop = [
    "generation hydro pumped storage aggregated",
    "forecast wind offshore eday ahead",
    "price actual",
    "price day ahead",
]
df_energy = df_energy.drop(columns=cols_to_drop)

# Convert to UTC-naive datetime and set as index
df_energy["time"] = pd.to_datetime(df_energy["time"], utc=True).dt.tz_convert(None)
df_energy = df_energy.set_index("time")

# [S1] Smart imputation for the target: shift(24) first, then linear interpolation
df_energy["total load actual"] = df_energy["total load actual"].fillna(
    df_energy["total load actual"].shift(24)
)
df_energy["total load actual"] = df_energy["total load actual"].interpolate(
    method="linear", limit_direction="forward"
)

# Linear interpolation for all other numeric columns
df_energy = df_energy.interpolate(method="linear", limit_direction="forward")

print(f"Energy dataset after cleaning: {df_energy.shape}")
print(f"Remaining missing values: {df_energy.isnull().sum().sum()}")


In [ ]:
df_energy.describe().round(2)


In [ ]:
df_energy.info()


In [ ]:

print('There are {} missing values or NaNs in df_energy.'
      .format(df_energy.isnull().values.sum()))

temp_energy = df_energy.duplicated(keep='first').sum()

print('There are {} duplicate rows in df_energy based on all columns.'
      .format(temp_energy))


## 1.2. Weather features dataset

In [ ]:
df_weather = pd.read_csv("../data/raw/weather_features.csv", parse_dates=["dt_iso"])


In [ ]:
df_weather.describe().round(2)


In [ ]:
print(f"Weather dataset shape: {df_weather.shape}")
print(f"Cities: {sorted(df_weather['city_name'].unique())}")
df_weather.head()


In [ ]:
print(f"Weather dataset shape: {df_weather.shape}")
print(f"Cities: {sorted(df_weather['city_name'].unique())}")
df_weather.head()


### Cleaning steps for the weather dataset

1. Rename `dt_iso` → `time` and convert to UTC-naive datetime.
2. **Drop duplicates** by (time, city_name) keeping the first occurrence (some hours have repeated weather records).
3. **Outliers in `pressure`:** replace values outside [931, 1051] hPa with NaN.
4. **Outliers in `wind_speed`:** replace values > 50 m/s with NaN.
5. Interpolate the resulting NaNs linearly.
6. Drop `rain_3h` (unreliable as documented in the source Kaggle notebook).
7. Drop the qualitative weather columns (`weather_main`, `weather_description`, `weather_id`, `weather_icon`).

In [ ]:
# Rename and convert
df_weather = df_weather.rename(columns={"dt_iso": "time"})
df_weather["time"] = pd.to_datetime(df_weather["time"], utc=True).dt.tz_convert(None)

# Drop duplicates by (time, city)
df_weather = df_weather.drop_duplicates(subset=["time", "city_name"], keep="first")

# Outliers in pressure and wind_speed → NaN
df_weather.loc[(df_weather["pressure"] > 1051) | (df_weather["pressure"] < 931), "pressure"] = np.nan
df_weather.loc[df_weather["wind_speed"] > 50, "wind_speed"] = np.nan

# Strip whitespace from city names (e.g. ' Barcelona' has a leading space)
df_weather["city_name"] = df_weather["city_name"].str.strip()

# Drop unreliable / qualitative columns
df_weather = df_weather.drop(columns=["rain_3h", "weather_main",
                                       "weather_description", "weather_id", "weather_icon"])

# Interpolate per city to preserve city-specific patterns
df_weather = df_weather.sort_values(["city_name", "time"])
num_cols = df_weather.select_dtypes(include=[np.number]).columns
df_weather[num_cols] = df_weather.groupby("city_name")[num_cols].transform(
    lambda x: x.interpolate(method="linear", limit_direction="both")
)

print(f"Weather dataset after cleaning: {df_weather.shape}")
print(f"Cities after stripping whitespace: {sorted(df_weather['city_name'].unique())}")


## 1.3. Merging the two datasets

In [ ]:
# Pivot weather: one column per (variable, city)
df_weather_pivot = df_weather.pivot_table(
    index="time",
    columns="city_name",
    values=num_cols
)
# Flatten multi-index columns: e.g. ('temp', 'Madrid') → 'temp_Madrid'
df_weather_pivot.columns = [f"{var}_{city}" for var, city in df_weather_pivot.columns]

# Merge with energy
df = df_energy.join(df_weather_pivot, how="left")

# Drop any remaining rows with missing target
df = df.dropna(subset=["total load actual"])

print(f"Merged dataset: {df.shape}")
print(f"Date range: {df.index.min()} → {df.index.max()}")


---
# 2. Visualizations and Time Series Analysis

This section explores patterns in the demand series. The findings will motivate the feature-engineering decisions in Section 3.

## 2.1. Useful visualizations and insights

In [ ]:
# Full demand series with weekly rolling mean
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df.index, df["total load actual"], color="#457B9D", alpha=0.4, lw=0.4, label="Hourly")
ax.plot(df.index, df["total load actual"].rolling(168).mean(),
        color="#E63946", lw=1.5, label="Weekly rolling mean")
ax.set_title("Figure 1 — Electricity Demand in Spain (2015–2018)",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Date"); ax.set_ylabel("Demand (MWh)")
ax.legend()
plt.tight_layout()
plt.savefig("../reports/figures/fig1_demand_series.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Zoom: 2 weeks (mid-January 2016) to see intraday and weekly patterns
zoom = df.loc["2016-01-11":"2016-01-25", "total load actual"]
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(zoom.index, zoom.values, color="#E63946", lw=1.2)
ax.set_title("Figure 2 — Demand Zoom: 2 Weeks (Jan 11–25, 2016)",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Date"); ax.set_ylabel("Demand (MWh)")
plt.tight_layout()
plt.savefig("../reports/figures/fig2_demand_zoom.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Daily, weekly and monthly profiles
df_tmp = df.copy()
df_tmp["hour"] = df_tmp.index.hour
df_tmp["weekday"] = df_tmp.index.dayofweek
df_tmp["month"] = df_tmp.index.month

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Hourly
hourly = df_tmp.groupby("hour")["total load actual"].mean()
axes[0].fill_between(hourly.index, hourly.values, alpha=0.25, color="#457B9D")
axes[0].plot(hourly.index, hourly.values, color="#457B9D", lw=2, marker="o", ms=4)
axes[0].set_title("Average Demand by Hour", fontweight="bold")
axes[0].set_xlabel("Hour"); axes[0].set_ylabel("Demand (MWh)")
axes[0].set_xticks(range(0, 24, 2))

# Weekday
weekday = df_tmp.groupby("weekday")["total load actual"].mean()
axes[1].bar(weekday.index, weekday.values, color="#2A9D8F", edgecolor="white")
axes[1].set_title("Average Demand by Weekday", fontweight="bold")
axes[1].set_xlabel("Weekday"); axes[1].set_ylabel("Demand (MWh)")
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(["Mon","Tue","Wed","Thu","Fri","Sat","Sun"])

# Monthly
monthly = df_tmp.groupby("month")["total load actual"].mean()
axes[2].plot(monthly.index, monthly.values, color="#E63946", lw=2, marker="o", ms=6)
axes[2].fill_between(monthly.index, monthly.values, alpha=0.2, color="#E63946")
axes[2].set_title("Average Demand by Month", fontweight="bold")
axes[2].set_xlabel("Month"); axes[2].set_ylabel("Demand (MWh)")
axes[2].set_xticks(range(1, 13))
axes[2].set_xticklabels(["J","F","M","A","M","J","J","A","S","O","N","D"])

plt.suptitle("Figure 3 — Demand Profiles by Hour, Weekday, and Month",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../reports/figures/fig3_profiles.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Histogram + temperature-vs-demand scatter
df_tmp["temp_mean"] = df_tmp[[f"temp_{c}" for c in ["Madrid","Barcelona","Valencia","Seville","Bilbao"]]].mean(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram
axes[0].hist(df_tmp["total load actual"], bins=80, color="#457B9D", edgecolor="white", lw=0.3)
axes[0].axvline(df_tmp["total load actual"].mean(), color="red", lw=1.5, ls="--",
                label=f"Mean = {df_tmp['total load actual'].mean():.0f}")
axes[0].set_title("Demand Distribution", fontweight="bold")
axes[0].set_xlabel("Demand (MWh)"); axes[0].set_ylabel("Frequency")
axes[0].legend()

# Temperature vs demand: expected inverted-U shape
sample = df_tmp.sample(min(5000, len(df_tmp)), random_state=SEED)
axes[1].scatter(sample["temp_mean"] - 273.15, sample["total load actual"],
                alpha=0.2, s=4, color="#E63946")
axes[1].set_title("Mean Temperature vs Demand (inverted-U pattern)",
                  fontweight="bold")
axes[1].set_xlabel("Mean Temperature (°C)"); axes[1].set_ylabel("Demand (MWh)")

plt.suptitle("Figure 4 — Demand Distribution and Temperature Effect",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../reports/figures/fig4_dist_temp.png", dpi=150, bbox_inches="tight")
plt.show()


**Key insights from the visualizations:**
- Strong daily pattern: morning ramp-up around 8–11 AM, evening peak around 8–10 PM.
- Weekly pattern: weekends show ~10–15% lower demand than weekdays.
- Seasonal pattern: peaks in summer (cooling) and winter (heating).
- Temperature–demand relation is approximately U-shaped → motivates the `temp²` feature [S5].

## 2.2. Decomposition and stationarity tests

In [ ]:
# Seasonal decomposition (weekly period = 24*7 = 168 hours)
ts = df["total load actual"].dropna()
decomp = seasonal_decompose(ts, model="additive", period=168)

fig, axes = plt.subplots(4, 1, figsize=(14, 9), sharex=True)
decomp.observed.plot(ax=axes[0], color="#1D3557", lw=0.5)
axes[0].set_ylabel("Observed")
decomp.trend.plot(ax=axes[1], color="#E63946", lw=1.5)
axes[1].set_ylabel("Trend")
decomp.seasonal.plot(ax=axes[2], color="#2A9D8F", lw=0.5)
axes[2].set_ylabel("Seasonal")
decomp.resid.plot(ax=axes[3], color="#999999", lw=0.4)
axes[3].set_ylabel("Residual"); axes[3].set_xlabel("Date")
plt.suptitle("Figure 5 — Seasonal Decomposition (weekly period = 168 h)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/figures/fig5_decomposition.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Stationarity tests on a sample (full series can be slow)
sample = ts.iloc[:8760]   # first year

adf  = adfuller(sample, autolag="AIC")
kp   = kpss(sample, regression="c", nlags="auto")

print("=== Stationarity Tests (first year of data) ===")
print(f"ADF  test:  statistic = {adf[0]:>8.3f}  | p-value = {adf[1]:.4f}  "
      f"→ {'STATIONARY' if adf[1] < 0.05 else 'NON-STATIONARY'} (reject H₀ if p < 0.05)")
print(f"KPSS test:  statistic = {kp[0]:>8.3f}  | p-value = {kp[1]:.4f}  "
      f"→ {'STATIONARY' if kp[1] > 0.05 else 'NON-STATIONARY'} (fail to reject H₀ if p > 0.05)")


## 2.3. Autocorrelation and partial autocorrelation

In [ ]:
sample = ts.iloc[:8760]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

plot_acf(sample, lags=200, ax=axes[0], color="#457B9D")
axes[0].set_title("Figure 6a — Autocorrelation Function (ACF)", fontweight="bold")

plot_pacf(sample, lags=50, ax=axes[1], color="#E63946", method="ywm")
axes[1].set_title("Figure 6b — Partial Autocorrelation (PACF)", fontweight="bold")

plt.tight_layout()
plt.savefig("../reports/figures/fig6_acf_pacf.png", dpi=150, bbox_inches="tight")
plt.show()


**Key insights from ACF/PACF:**
- Strong autocorrelation at lag 1 → previous hour.
- Strong autocorrelation at lag 24 → same hour previous day.
- Strong autocorrelation at lag 168 → same hour previous week.

These three lags motivate the lag features in Section 3.

---
# 3. Feature Engineering

We engineer a rich set of features following the strategy: time features (with cyclic encoding [S4]), lag features (motivated by ACF/PACF), rolling features, temperature transformations (including `temp²` [S5]), Spanish holidays [S3], year trend [S6], and interactions [S8].

## 3.1. Feature generation

### Temporal features + cyclic encoding [S4]

In [ ]:
df["hour"]       = df.index.hour
df["weekday"]    = df.index.dayofweek
df["month"]      = df.index.month
df["is_weekend"] = (df.index.dayofweek >= 5).astype(int)
df["year"]       = df.index.year   # [S6] capture multi-year trend

# Business hour pattern (Spanish siesta)
def get_business_hour(h):
    if (9 <= h < 14) or (17 <= h < 21):
        return 2   # working hours
    elif 14 <= h < 17:
        return 1   # siesta
    return 0       # off-hours
df["business_hour"] = df["hour"].map(get_business_hour)

# [S4] Cyclic encoding (sin/cos) — preserves distance between hour 23 and hour 0
df["hour_sin"]    = np.sin(2 * np.pi * df["hour"]    / 24)
df["hour_cos"]    = np.cos(2 * np.pi * df["hour"]    / 24)
df["weekday_sin"] = np.sin(2 * np.pi * df["weekday"] / 7)
df["weekday_cos"] = np.cos(2 * np.pi * df["weekday"] / 7)
df["month_sin"]   = np.sin(2 * np.pi * df["month"]   / 12)
df["month_cos"]   = np.cos(2 * np.pi * df["month"]   / 12)

print("✅ Time features + cyclic encoding done")


### Spanish holidays [S3]

In [ ]:
es_holidays = holidays.Spain(years=list(range(2015, 2019)))
df["is_holiday"] = df.index.normalize().isin(es_holidays).astype(int)

# Day type: 0–4 weekday, 5 Saturday, 6 Sunday, 7 Holiday
df["day_type"] = df["weekday"].copy()
df.loc[df["is_holiday"] == 1, "day_type"] = 7

print(f"Holidays detected: {df['is_holiday'].sum():,} hours "
      f"({df['is_holiday'].sum()/24:.0f} days)")


### Lag features (motivated by ACF/PACF)

In [ ]:
TARGET = "total load actual"
df["load_lag_1"]   = df[TARGET].shift(1)
df["load_lag_24"]  = df[TARGET].shift(24)
df["load_lag_168"] = df[TARGET].shift(168)
print("✅ Lag features done")


### Rolling features

In [ ]:
# shift(1) before rolling to avoid leakage from the current hour
df["load_roll_24h_mean"] = df[TARGET].shift(1).rolling(24).mean()
df["load_roll_24h_std"]  = df[TARGET].shift(1).rolling(24).std()
df["load_roll_7d_mean"]  = df[TARGET].shift(1).rolling(168).mean()
print("✅ Rolling features done")


### Temperature features (weighted + quadratic [S5] + range)

In [ ]:
cities  = ["Madrid", "Barcelona", "Valencia", "Seville", "Bilbao"]
weights = {"Madrid":0.405, "Barcelona":0.341, "Valencia":0.108, "Seville":0.086, "Bilbao":0.065}

# Population-weighted mean temperature across the 5 cities
df["temp_weighted"] = sum(df[f"temp_{c}"] * w for c, w in weights.items())

# [S5] Quadratic temperature — captures inverted-U (heating + cooling)
df["temp_weighted_sq"] = df["temp_weighted"] ** 2

# Temperature range per city (max - min)
for c in cities:
    df[f"temp_range_{c}"] = df[f"temp_max_{c}"] - df[f"temp_min_{c}"]

# Average humidity and wind speed across cities
df["humidity_mean"]   = df[[f"humidity_{c}"   for c in cities]].mean(axis=1)
df["wind_speed_mean"] = df[[f"wind_speed_{c}" for c in cities]].mean(axis=1)

print("✅ Temperature & weather aggregations done")


### Interactions [S8]

In [ ]:
df["temp_x_daytype"]    = df["temp_weighted"]    * df["day_type"]
df["temp_sq_x_weekend"]  = df["temp_weighted_sq"] * df["is_weekend"]
print("✅ Interaction features done")


## 3.2. Feature selection [S2]

We deliberately **drop** several feature groups for a demand-prediction task:

| Group | Reason |
|---|---|
| All `generation_*` columns | Reverse causality: demand is met by generation, not the other way around. Including them is conceptual leakage. |
| `wind_deg_*` (5 cities) | No physical relation to consumption. |
| `pressure_*` (5 cities) | Near-zero correlation with demand. |
| `rain_1h_*` (5 cities) | Very weak signal; mostly noise. |
| `clouds_all_*` (5 cities) | Effect already captured by temperature; introduces multicollinearity. |
| `temp_min_*`, `temp_max_*` per city | Replaced by `temp_range_*`. |
| `snow_3h_*` (5 cities) | Mostly zero in Spain; not useful. |
| `total load forecast` | It is itself a forecast → would dominate without learning anything. |

In [ ]:
# Build drop list
gen_cols    = [c for c in df.columns if c.startswith("generation")]
wind_deg    = [f"wind_deg_{c}"    for c in cities]
pressure_c  = [f"pressure_{c}"    for c in cities]
rain_1h_c   = [f"rain_1h_{c}"     for c in cities]
clouds_c    = [f"clouds_all_{c}"  for c in cities]
temp_min_c  = [f"temp_min_{c}"    for c in cities]
temp_max_c  = [f"temp_max_{c}"    for c in cities]
snow_c      = [f"snow_3h_{c}"     for c in cities if f"snow_3h_{c}" in df.columns]

drop_extra = gen_cols + wind_deg + pressure_c + rain_1h_c + clouds_c \
             + temp_min_c + temp_max_c + snow_c + ["total load forecast"]

drop_extra = [c for c in drop_extra if c in df.columns]
df = df.drop(columns=drop_extra)

# Drop initial rows with NaN from lag features
df_model = df.dropna()

print(f"Final feature space: {df_model.shape[1] - 1} features × {df_model.shape[0]:,} samples")
print(f"Dropped {len(drop_extra)} non-causal / redundant columns")


---
# 4. Electricity Demand Forecasting

We compare a **Naive baseline** against **7 models** from different families:
- Linear: Ridge
- Non-parametric: KNN
- Kernel: Linear SVR (much faster than RBF SVR on 35k rows)
- Bagging: Random Forest
- Boosting (sklearn): Gradient Boosting
- Boosting (optimized): XGBoost
- Neural: Multi-Layer Perceptron (MLP)

Then we combine them with a **weighted average**, where the weights are calculated from validation RMSE.

### Chronological split (75 / 12 / 13)

In [ ]:
X = df_model.drop(columns=[TARGET])
y = df_model[TARGET]

n = len(df_model)
train_end = int(n * 0.75)
val_end   = int(n * 0.875)

X_train, y_train = X.iloc[:train_end],         y.iloc[:train_end]
X_val,   y_val   = X.iloc[train_end:val_end],  y.iloc[train_end:val_end]
X_test,  y_test  = X.iloc[val_end:],           y.iloc[val_end:]

# Scaled versions for Ridge / KNN / SVR / MLP
scaler   = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

print(f"Train : {len(X_train):,} samples ({len(X_train)/n*100:.1f}%)")
print(f"Val   : {len(X_val):,}  samples ({len(X_val)/n*100:.1f}%)")
print(f"Test  : {len(X_test):,}  samples ({len(X_test)/n*100:.1f}%)")
print(f"\nTrain : {X_train.index.min().date()} → {X_train.index.max().date()}")
print(f"Val   : {X_val.index.min().date()}   → {X_val.index.max().date()}")
print(f"Test  : {X_test.index.min().date()}  → {X_test.index.max().date()}")

# Container for all predictions
results = {}


## 4.1. Naive Forecast

Baseline: predict each hour's demand as the demand 24 hours earlier (same hour previous day).

In [ ]:
y_pred_naive_val  = X_val["load_lag_24"].values
y_pred_naive_test = X_test["load_lag_24"].values

rmse_naive = sqrt(mean_squared_error(y_test, y_pred_naive_test))
print(f"Naive  →  Test RMSE: {rmse_naive:,.1f} MWh")

results["Naive"] = (y_pred_naive_val, y_pred_naive_test)


## 4.2. Ridge Regression

Linear regression with L2 regularization. Strong baseline for many correlated features.

In [ ]:
ridge = Ridge(alpha=10.0, random_state=SEED)
ridge.fit(X_train_s, y_train)

y_pred_ridge_val  = ridge.predict(X_val_s)
y_pred_ridge_test = ridge.predict(X_test_s)

rmse_ridge = sqrt(mean_squared_error(y_test, y_pred_ridge_test))
print(f"Ridge  →  Test RMSE: {rmse_ridge:,.1f} MWh")

results["Ridge"] = (y_pred_ridge_val, y_pred_ridge_test)


## 4.3. K-Nearest Neighbors (KNN)

Non-parametric model. Predicts based on the most similar historical observations.

In [ ]:
knn = KNeighborsRegressor(n_neighbors=10, weights="distance", n_jobs=-1)
knn.fit(X_train_s, y_train)

y_pred_knn_val  = knn.predict(X_val_s)
y_pred_knn_test = knn.predict(X_test_s)

rmse_knn = sqrt(mean_squared_error(y_test, y_pred_knn_test))
print(f"KNN    →  Test RMSE: {rmse_knn:,.1f} MWh")

results["KNN"] = (y_pred_knn_val, y_pred_knn_test)


## 4.4. Linear SVR

Support Vector Regression with a linear kernel. We use `LinearSVR` (much faster than RBF SVR for 35k rows) — the strategy document explicitly allows this swap.

In [ ]:
svr = LinearSVR(C=1.0, epsilon=50, max_iter=5000, random_state=SEED)
svr.fit(X_train_s, y_train)

y_pred_svr_val  = svr.predict(X_val_s)
y_pred_svr_test = svr.predict(X_test_s)

rmse_svr = sqrt(mean_squared_error(y_test, y_pred_svr_test))
print(f"SVR    →  Test RMSE: {rmse_svr:,.1f} MWh")

results["SVR"] = (y_pred_svr_val, y_pred_svr_test)


## 4.5. Random Forest

Bagging of decision trees. Robust to outliers and naturally handles non-linear interactions.

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200, max_depth=20, min_samples_leaf=5,
    max_features=0.7, n_jobs=-1, random_state=SEED
)
rf.fit(X_train, y_train)   # tree-based → no scaling needed

y_pred_rf_val  = rf.predict(X_val)
y_pred_rf_test = rf.predict(X_test)

rmse_rf = sqrt(mean_squared_error(y_test, y_pred_rf_test))
print(f"RandomForest  →  Test RMSE: {rmse_rf:,.1f} MWh")

results["RF"] = (y_pred_rf_val, y_pred_rf_test)


## 4.6. Gradient Boosting (sklearn)

Sequential boosting of shallow trees. Different implementation philosophy than XGBoost.

In [ ]:
gb = GradientBoostingRegressor(
    n_estimators=200, learning_rate=0.05, max_depth=4,
    subsample=0.8, random_state=SEED
)
gb.fit(X_train, y_train)

y_pred_gb_val  = gb.predict(X_val)
y_pred_gb_test = gb.predict(X_test)

rmse_gb = sqrt(mean_squared_error(y_test, y_pred_gb_test))
print(f"GradBoost  →  Test RMSE: {rmse_gb:,.1f} MWh")

results["GradBoost"] = (y_pred_gb_val, y_pred_gb_test)


## 4.7. XGBoost

Optimized boosting with L1+L2 regularization and early stopping.

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1,
    objective="reg:squarederror",
    random_state=SEED, n_jobs=-1,
    early_stopping_rounds=20, verbosity=0
)
xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

y_pred_xgb_val  = xgb_model.predict(X_val)
y_pred_xgb_test = xgb_model.predict(X_test)

rmse_xgb = sqrt(mean_squared_error(y_test, y_pred_xgb_test))
print(f"XGBoost  →  Test RMSE: {rmse_xgb:,.1f} MWh")

results["XGBoost"] = (y_pred_xgb_val, y_pred_xgb_test)


## 4.8. Multi-Layer Perceptron (MLP)

Feed-forward neural network. Approximates arbitrary non-linear functions.

In [ ]:
mlp = MLPRegressor(
    hidden_layer_sizes=(128, 64, 32),
    activation="relu", max_iter=300,
    early_stopping=True, validation_fraction=0.1,
    random_state=SEED
)
mlp.fit(X_train_s, y_train)

y_pred_mlp_val  = mlp.predict(X_val_s)
y_pred_mlp_test = mlp.predict(X_test_s)

rmse_mlp = sqrt(mean_squared_error(y_test, y_pred_mlp_test))
print(f"MLP  →  Test RMSE: {rmse_mlp:,.1f} MWh")

results["MLP"] = (y_pred_mlp_val, y_pred_mlp_test)


## 4.9. Ensemble — Weighted Average

We combine the 7 models (Naive is excluded — it is only a reference baseline). Two weighting schemes:

1. **1/RMSE heuristic** — each model's weight is inversely proportional to its validation RMSE.
2. **[S7] scipy-optimized weights** — directly minimize the ensemble validation RMSE.

In [ ]:
# Drop the Naive entry — it is the baseline, not part of the ensemble
ensemble_models = {k: v for k, v in results.items() if k != "Naive"}

# Step 1 — validation RMSE per model
rmse_val = {name: sqrt(mean_squared_error(y_val, vp))
            for name, (vp, _) in ensemble_models.items()}

print("Validation RMSE per model:")
for name, r in sorted(rmse_val.items(), key=lambda x: x[1]):
    print(f"  {name:11s}: {r:>8,.1f} MWh")


In [ ]:
# Step 2A — Heuristic: weights ∝ 1/RMSE
inv = {name: 1/r for name, r in rmse_val.items()}
total = sum(inv.values())
weights_heuristic = {name: v/total for name, v in inv.items()}

print("\nHeuristic ensemble weights (1/RMSE):")
for name, w in sorted(weights_heuristic.items(), key=lambda x: -x[1]):
    print(f"  {name:11s}: {w:.1%}")


In [ ]:
# Step 2B — [S7] Optimize weights with scipy to directly minimize ensemble RMSE on validation
val_preds = np.array([vp for vp, _ in ensemble_models.values()])

def ensemble_rmse(w):
    w_norm = np.abs(w) / np.abs(w).sum()   # force positive weights summing to 1
    y_pred = np.dot(w_norm, val_preds)
    return sqrt(mean_squared_error(y_val, y_pred))

x0 = np.ones(len(ensemble_models)) / len(ensemble_models)
res = minimize(ensemble_rmse, x0=x0, method="Nelder-Mead",
               options={"xatol": 1e-4, "maxiter": 5000})

weights_optim = np.abs(res.x) / np.abs(res.x).sum()
names = list(ensemble_models.keys())

print("\nOptimized ensemble weights (scipy Nelder-Mead):")
for name, w in sorted(zip(names, weights_optim), key=lambda x: -x[1]):
    print(f"  {name:11s}: {w:.1%}")


In [ ]:
# Step 3 — Build ensemble predictions on test set
test_preds = np.array([tp for _, tp in ensemble_models.values()])

# Heuristic ensemble
w_h = np.array([weights_heuristic[n] for n in names])
y_pred_ens_h = np.dot(w_h, test_preds)

# Optimized ensemble
y_pred_ens_o = np.dot(weights_optim, test_preds)

# Simple average (for reference)
y_pred_ens_s = test_preds.mean(axis=0)

rmse_ens_h = sqrt(mean_squared_error(y_test, y_pred_ens_h))
rmse_ens_o = sqrt(mean_squared_error(y_test, y_pred_ens_o))
rmse_ens_s = sqrt(mean_squared_error(y_test, y_pred_ens_s))

print(f"Ensemble (heuristic 1/RMSE)  →  Test RMSE: {rmse_ens_h:,.1f} MWh")
print(f"Ensemble (optimized)         →  Test RMSE: {rmse_ens_o:,.1f} MWh")
print(f"Ensemble (simple average)    →  Test RMSE: {rmse_ens_s:,.1f} MWh")


---
# 5. Results

Final comparison across all models and ensemble variants.

In [ ]:
def evaluate(name, y_true, y_pred):
    rmse = sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2   = r2_score(y_true, y_pred)
    return {"Model": name, "RMSE (MWh)": rmse, "MAE (MWh)": mae,
            "MAPE (%)": mape, "R²": r2}

rows = [
    evaluate("Naive (t-24)",          y_test, y_pred_naive_test),
    evaluate("Ridge",                 y_test, y_pred_ridge_test),
    evaluate("KNN",                   y_test, y_pred_knn_test),
    evaluate("SVR (linear)",          y_test, y_pred_svr_test),
    evaluate("Random Forest",         y_test, y_pred_rf_test),
    evaluate("Gradient Boosting",     y_test, y_pred_gb_test),
    evaluate("XGBoost",               y_test, y_pred_xgb_test),
    evaluate("MLP",                   y_test, y_pred_mlp_test),
    evaluate("Ensemble heuristic",    y_test, y_pred_ens_h),
    evaluate("Ensemble optimized",    y_test, y_pred_ens_o),
    evaluate("Ensemble simple avg",   y_test, y_pred_ens_s),
]

df_results = pd.DataFrame(rows).sort_values("RMSE (MWh)").reset_index(drop=True)
df_results.to_csv("../results/results_table.csv", index=False)

# Color-coded display
def highlight(row):
    if "Ensemble" in row["Model"]:
        return ["background-color: #fff3cd"] * len(row)
    if "Naive" in row["Model"]:
        return ["background-color: #f0f0f0"] * len(row)
    return [""] * len(row)

(df_results.style
    .apply(highlight, axis=1)
    .format({"RMSE (MWh)":"{:,.1f}", "MAE (MWh)":"{:,.1f}",
             "MAPE (%)":"{:.2f}", "R²":"{:.4f}"})
    .set_caption("Final results — all models on the test set (lower RMSE = better)"))


In [ ]:
# Figure 7 — RMSE bar chart (all models)
df_plot = df_results.copy()

colors = []
for m in df_plot["Model"]:
    if "Ensemble" in m:    colors.append("#F77F00")
    elif "Naive" in m:     colors.append("#999999")
    else:                  colors.append("#457B9D")

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(df_plot["Model"], df_plot["RMSE (MWh)"],
               color=colors, edgecolor="white", lw=0.4)
for bar, v in zip(bars, df_plot["RMSE (MWh)"]):
    ax.text(v + 20, bar.get_y() + bar.get_height()/2,
            f"{v:,.0f}", va="center", fontsize=9)
ax.set_xlabel("RMSE (MWh) — lower is better")
ax.set_title("Figure 7 — Test RMSE Comparison Across Models",
             fontsize=12, fontweight="bold")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("../reports/figures/fig7_rmse_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Figure 8 — Ensemble weights (pie chart)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

w_h_list = [weights_heuristic[n] for n in names]
axes[0].pie(w_h_list, labels=names, autopct="%1.1f%%", startangle=90,
            colors=sns.color_palette("Set2", len(names)))
axes[0].set_title("Heuristic Weights (1/RMSE)", fontweight="bold")

axes[1].pie(weights_optim, labels=names, autopct="%1.1f%%", startangle=90,
            colors=sns.color_palette("Set2", len(names)))
axes[1].set_title("Optimized Weights (scipy)", fontweight="bold")

plt.suptitle("Figure 8 — Ensemble Weight Distributions",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../reports/figures/fig8_ensemble_weights.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Figure 9 — Predictions vs actual (2-week zoom) for best ensemble
best_ens = y_pred_ens_o if rmse_ens_o < rmse_ens_h else y_pred_ens_h

# Pick the first 2 weeks of the test set
zoom_n   = 24 * 14
zoom_idx = y_test.index[:zoom_n]

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(zoom_idx, y_test.values[:zoom_n], color="#1D3557", lw=1.5, label="Actual")
ax.plot(zoom_idx, best_ens[:zoom_n],      color="#E63946", lw=1.2, ls="--",
        label="Ensemble forecast")
ax.set_title("Figure 9 — Ensemble Forecast vs Actual (first 2 test weeks)",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Date"); ax.set_ylabel("Demand (MWh)")
ax.legend()
plt.tight_layout()
plt.savefig("../reports/figures/fig9_forecast_vs_actual.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Figure 10 — Feature importance (XGBoost)
imp = pd.Series(xgb_model.feature_importances_,
                index=X_train.columns).sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 7))
imp[::-1].plot(kind="barh", color="#E63946", edgecolor="white", alpha=0.85)
ax.set_title("Figure 10 — Top 20 Feature Importances (XGBoost)",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Importance (gain)")
plt.tight_layout()
plt.savefig("../reports/figures/fig10_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Figure 11 — Residual analysis of the best ensemble
residuals = y_test.values - best_ens

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(residuals, bins=60, color="#457B9D", edgecolor="white", lw=0.3)
axes[0].axvline(0, color="red", lw=1.5, ls="--")
axes[0].set_title("Residual Distribution", fontweight="bold")
axes[0].set_xlabel("Residual (MWh)"); axes[0].set_ylabel("Frequency")

axes[1].scatter(best_ens, residuals, alpha=0.1, s=4, color="#E63946")
axes[1].axhline(0, color="black", lw=1.2, ls="--")
axes[1].set_title("Residuals vs Predicted", fontweight="bold")
axes[1].set_xlabel("Predicted (MWh)"); axes[1].set_ylabel("Residual")

mn, mx = y_test.min(), y_test.max()
axes[2].scatter(y_test, best_ens, alpha=0.1, s=4, color="#2A9D8F")
axes[2].plot([mn, mx], [mn, mx], "r--", lw=1.5, label="Perfect fit")
axes[2].set_title("Actual vs Predicted", fontweight="bold")
axes[2].set_xlabel("Actual (MWh)"); axes[2].set_ylabel("Predicted (MWh)")
axes[2].legend()

best_rmse = min(rmse_ens_h, rmse_ens_o)
plt.suptitle(f"Figure 11 — Residual Analysis · Best Ensemble · Test RMSE = {best_rmse:,.1f} MWh",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../reports/figures/fig11_residuals.png", dpi=150, bbox_inches="tight")
plt.show()


---
# 6. References

1. **Dietterich, T. G. (2000).** Ensemble methods in machine learning. *International Workshop on Multiple Classifier Systems*, 1–15.
2. **Breiman, L. (1996).** Bagging predictors. *Machine Learning*, 24(2), 123–140.
3. **Breiman, L. (2001).** Random forests. *Machine Learning*, 45(1), 5–32.
4. **Friedman, J. H. (2001).** Greedy function approximation: a gradient boosting machine. *Annals of Statistics*, 29(5), 1189–1232.
5. **Chen, T., & Guestrin, C. (2016).** XGBoost: a scalable tree boosting system. *Proceedings of the 22nd ACM SIGKDD*, 785–794.
6. **Hong, T., & Fan, S. (2016).** Probabilistic electric load forecasting: a tutorial review. *International Journal of Forecasting*, 32(3), 914–938.
7. **Hippert, H. S., Pedreira, C. E., & Souza, R. C. (2001).** Neural networks for short-term load forecasting: a review and evaluation. *IEEE Transactions on Power Systems*, 16(1), 44–55.
8. **Chou, J.-S., & Tran, D.-S. (2018).** Forecasting energy consumption time series using machine learning techniques based on usage patterns of residential householders. *Energy*, 165, 709–726.
9. **Lago, J., De Ridder, F., & De Schutter, B. (2018).** Forecasting spot electricity prices: deep learning approaches and empirical comparison of traditional algorithms. *Applied Energy*, 221, 386–405.